# 🧠 MSI-Net V2 — VGG16 + FPN Decoder (Surgical Improvement)

**Phân tích & chiến lược cải tiến:**

Qua so sánh baseline và phiên bản trước, nguyên nhân gốc rễ là:
1. **Thay backbone** (VGG16-Hybrid → EfficientNet) làm mất domain fit (Places365 scene context)
2. **Bỏ MSI** (multi-scale input) — mất inductive bias cốt lõi
3. **LR quá lớn** (3e-4 thay vì 1e-5) — phá vỡ pretrained weights
4. **Data pipeline** không tương đương (bilinear resize vs aspect-ratio + symmetric pad)

**Chiến lược V2 — chỉ thay một thứ có khả năng cải thiện cao nhất:**
> Giữ nguyên VGG16-Hybrid backbone + MSI + data pipeline + lr=1e-5  
> Chỉ nâng cấp **decoder**: thay UpSampling 8× đơn giản bằng **FPN-style với skip connections từ conv3/conv4/conv5**

Lý do chọn decoder upgrade:
- Không đụng backbone → giữ domain fit VGG-Hybrid
- Không đụng MSI → giữ multi-scale input
- Skip connections từ intermediate layers giúp recover spatial detail bị mất qua MaxPool
- Rủi ro thấp, gain rõ ràng theo lý thuyết FPN


## ── Bước 1: Cài đặt thư viện ────────────────────

In [ ]:
%%capture
!pip install gdown h5py scipy matplotlib tensorflow==2.1 requests

In [ ]:
## ── GPU Sanity Check ──────────────────────────────────
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ Không có GPU! Vào Settings → Accelerator → GPU T4 rồi chạy lại.")
print("✅ GPU OK:")
print(result.stdout[:300])


## ── Bước 2: Khởi tạo thư mục làm việc ────────────────

In [ ]:
import os, sys
BASE_DIR    = "/kaggle/working"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
WEIGHTS_DIR = os.path.join(BASE_DIR, "weights")
for d in [DATA_DIR, RESULTS_DIR, WEIGHTS_DIR,
          os.path.join(RESULTS_DIR, "history"),
          os.path.join(RESULTS_DIR, "images"),
          os.path.join(RESULTS_DIR, "ckpts", "best")]:
    os.makedirs(d, exist_ok=True)
print("✅ Thư mục tạo xong")

## ── Bước 3: Định nghĩa các module ──────────────────

In [ ]:
%%writefile /kaggle/working/config.py
PARAMS = {
    "n_epochs":      10,
    "batch_size":    4,
    "learning_rate": 1e-5,   # Giữ nguyên như baseline — critical!
    "device":        "gpu",
}
DIMS = {
    "image_size_salicon":  (240, 320),
    "image_size_mit1003":  (360, 360),
    "image_size_cat2000":  (216, 384),
    "image_size_dutomron": (360, 360),
    "image_size_pascals":  (360, 360),
    "image_size_osie":     (240, 320),
    "image_size_fiwi":     (216, 384),
}

In [ ]:
%%writefile /kaggle/working/loss.py
import tensorflow as tf

def kld(y_true, y_pred, eps=1e-7):
    """KL Divergence loss — giống hệt baseline."""
    sum_per_image = tf.reduce_sum(y_true, axis=(1,2,3), keepdims=True)
    y_true = y_true / (eps + sum_per_image)
    sum_per_image = tf.reduce_sum(y_pred, axis=(1,2,3), keepdims=True)
    y_pred = y_pred / (eps + sum_per_image)
    loss = y_true * tf.math.log(eps + y_true / (eps + y_pred))
    return tf.reduce_mean(tf.reduce_sum(loss, axis=(1,2,3)))

In [ ]:
%%writefile /kaggle/working/model.py
"""
MSI-Net V2: VGG16-Hybrid Encoder + FPN-style Decoder với skip connections.

Thay đổi so với baseline:
- Encoder: giữ nguyên VGG16-style (conv1→conv5), load pretrained VGG16-Hybrid weights
- MSI: giữ nguyên 3 nhánh encoder song song (full, 1/2, 1/4)
- Decoder: NÂNG CẤP từ UpSampling 8× → FPN-style với skip connections
    * concat features từ 3 MSI branches (1536ch) → bottleneck Conv 512
    * skip từ pool3 level (avg của 3 nhánh, 256ch) → fuse sau up2x
    * skip từ pool2 level (avg của 3 nhánh, 128ch) → fuse sau up4x
    * skip từ pool1 level (avg của 3 nhánh, 64ch)  → fuse sau up8x
    * Conv 1×1 sigmoid → resize to input
"""
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np, os

VGG_LAYER_NAMES = [
    'conv1_1','conv1_2',
    'conv2_1','conv2_2',
    'conv3_1','conv3_2','conv3_3',
    'conv4_1','conv4_2','conv4_3',
    'conv5_1','conv5_2','conv5_3',
]

def _vgg_encoder(inp, suffix=''):
    """VGG16-style encoder — giữ nguyên như baseline."""
    def conv(x, f, name):
        return layers.Conv2D(f, 3, activation='relu', padding='same',
                             name=name+suffix)(x)
    def pool(x, name):
        return layers.MaxPooling2D(2, 2, padding='same', name=name+suffix)(x)

    x = conv(inp, 64,  'conv1_1'); x = conv(x, 64,  'conv1_2')
    p1 = pool(x, 'pool1')   # /2  — 64ch  → skip level 1
    x = conv(p1, 128, 'conv2_1'); x = conv(x, 128, 'conv2_2')
    p2 = pool(x, 'pool2')   # /4  — 128ch → skip level 2
    x = conv(p2, 256, 'conv3_1'); x = conv(x, 256, 'conv3_2')
    x = conv(x,  256, 'conv3_3')
    p3 = pool(x, 'pool3')   # /8  — 256ch → skip level 3
    x = conv(p3, 512, 'conv4_1'); x = conv(x, 512, 'conv4_2')
    x = conv(x,  512, 'conv4_3')
    p4 = pool(x, 'pool4')   # /16 — 512ch
    x = conv(p4, 512, 'conv5_1'); x = conv(x, 512, 'conv5_2')
    x = conv(x,  512, 'conv5_3')  # /16 — 512ch (no pool5)
    return x, p3, p2, p1   # feat_out, skip3, skip2, skip1

def _fpn_block(x, skip, filters, name, target_h, target_w):
    """FPN decode block: upsample x 2×, fuse with skip, refine."""
    x = layers.UpSampling2D(size=(2, 2), interpolation='bilinear',
                             name=f'up_{name}')(x)
    # Resize skip về đúng shape của x (tránh off-by-one)
    x = layers.Resizing(target_h, target_w, name=f'resize_x_{name}')(x)
    skip_r = layers.Resizing(target_h, target_w, name=f'resize_skip_{name}')(skip)
    skip_r = layers.Conv2D(filters, 1, padding='same', use_bias=False,
                           name=f'proj_skip_{name}')(skip_r)
    skip_r = layers.Activation('relu', name=f'act_skip_{name}')(skip_r)
    x = layers.Add(name=f'add_{name}')([x[:,:,:,:filters] if x.shape[-1]==filters
                                         else layers.Conv2D(filters, 1, padding='same',
                                                use_bias=False, name=f'proj_x_{name}')(x),
                                         skip_r])
    x = layers.Conv2D(filters, 3, padding='same', activation='relu',
                      use_bias=False, name=f'refine_{name}')(x)
    return x

def build_msinet(input_shape=(240, 320, 3)):
    """
    MSI-Net V2 với FPN decoder.
    Giữ nguyên: encoder VGG16-style + 3-branch MSI.
    Nâng cấp: decoder FPN với skip connections từ pool1/2/3.
    """
    h, w = input_shape[0], input_shape[1]

    # ── 3 inputs ────────────────────────────────────────────────────
    inp_full    = tf.keras.Input(shape=input_shape,          name='input_full')
    inp_half    = tf.keras.Input(shape=(h//2, w//2, 3),      name='input_half')
    inp_quarter = tf.keras.Input(shape=(h//4, w//4, 3),      name='input_quarter')

    # ── 3 nhánh VGG encoder — mỗi nhánh trả về (feat, skip3, skip2, skip1) ─
    feat_f, s3_f, s2_f, s1_f = _vgg_encoder(inp_full,    suffix='')
    feat_h, s3_h, s2_h, s1_h = _vgg_encoder(inp_half,    suffix='_half')
    feat_q, s3_q, s2_q, s1_q = _vgg_encoder(inp_quarter, suffix='_quarter')

    # ── Align half & quarter features về cùng spatial với full ─────
    # Shape full feat: (h//16, w//16) — dùng resize dynamic
    feat_h_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_feat_half')([feat_h, feat_f])
    feat_q_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_feat_quarter')([feat_q, feat_f])

    # ── Concat 3 branches → 1536ch → bottleneck 512 ────────────────
    merged = layers.Concatenate(name='concat_scales')(
                [feat_f, feat_h_up, feat_q_up])
    x = layers.Conv2D(512, 1, padding='same', activation='relu',
                      name='bottleneck')(merged)

    # ── Skip connections: average pool-level features từ 3 nhánh ───
    # pool3-level: /8 → 256ch
    s3_h_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s3_half')([s3_h, s3_f])
    s3_q_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s3_quarter')([s3_q, s3_f])
    skip3 = layers.Average(name='avg_skip3')([s3_f, s3_h_up, s3_q_up])

    # pool2-level: /4 → 128ch
    s2_h_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s2_half')([s2_h, s2_f])
    s2_q_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s2_quarter')([s2_q, s2_f])
    skip2 = layers.Average(name='avg_skip2')([s2_f, s2_h_up, s2_q_up])

    # pool1-level: /2 → 64ch
    s1_h_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s1_half')([s1_h, s1_f])
    s1_q_up = layers.Lambda(
        lambda t: tf.image.resize(t[0], tf.shape(t[1])[1:3]),
        name='resize_s1_quarter')([s1_q, s1_f])
    skip1 = layers.Average(name='avg_skip1')([s1_f, s1_h_up, s1_q_up])

    # ── FPN Decoder ──────────────────────────────────────────────────
    # x: (h//16, w//16, 512) → up 2× → (h//8, w//8)
    x = layers.UpSampling2D(size=(2,2), interpolation='bilinear', name='up_d4')(x)
    x = layers.Resizing(h//8, w//8, name='resize_d4')(x)
    skip3_p = layers.Conv2D(256, 1, padding='same', activation='relu',
                             use_bias=False, name='proj_skip3')(skip3)
    skip3_r = layers.Resizing(h//8, w//8, name='resize_skip3')(skip3_p)
    x_proj  = layers.Conv2D(256, 1, padding='same', use_bias=False, name='proj_x_d4')(x)
    x = layers.Add(name='add_d4')([x_proj, skip3_r])
    x = layers.Conv2D(256, 3, padding='same', activation='relu',
                      use_bias=False, name='refine_d4')(x)

    # x: (h//8, w//8, 256) → up 2× → (h//4, w//4)
    x = layers.UpSampling2D(size=(2,2), interpolation='bilinear', name='up_d3')(x)
    x = layers.Resizing(h//4, w//4, name='resize_d3')(x)
    skip2_p = layers.Conv2D(128, 1, padding='same', activation='relu',
                             use_bias=False, name='proj_skip2')(skip2)
    skip2_r = layers.Resizing(h//4, w//4, name='resize_skip2')(skip2_p)
    x_proj  = layers.Conv2D(128, 1, padding='same', use_bias=False, name='proj_x_d3')(x)
    x = layers.Add(name='add_d3')([x_proj, skip2_r])
    x = layers.Conv2D(128, 3, padding='same', activation='relu',
                      use_bias=False, name='refine_d3')(x)

    # x: (h//4, w//4, 128) → up 2× → (h//2, w//2)
    x = layers.UpSampling2D(size=(2,2), interpolation='bilinear', name='up_d2')(x)
    x = layers.Resizing(h//2, w//2, name='resize_d2')(x)
    skip1_p = layers.Conv2D(64, 1, padding='same', activation='relu',
                             use_bias=False, name='proj_skip1')(skip1)
    skip1_r = layers.Resizing(h//2, w//2, name='resize_skip1')(skip1_p)
    x_proj  = layers.Conv2D(64, 1, padding='same', use_bias=False, name='proj_x_d2')(x)
    x = layers.Add(name='add_d2')([x_proj, skip1_r])
    x = layers.Conv2D(64, 3, padding='same', activation='relu',
                      use_bias=False, name='refine_d2')(x)

    # x: (h//2, w//2, 64) → up 2× → full res
    x = layers.UpSampling2D(size=(2,2), interpolation='bilinear', name='up_d1')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu',
                      use_bias=False, name='refine_d1')(x)
    x = layers.Conv2D(1, 1, activation='sigmoid', padding='same', name='output')(x)
    out = layers.Resizing(h, w, name='resize_output')(x)

    model = Model(inputs=[inp_full, inp_half, inp_quarter],
                  outputs=out, name='MSINet_V2_FPN')
    return model


def load_vgg16_hybrid_weights(model, weights_path):
    """
    Load pretrained VGG16-Hybrid weights vào cả 3 nhánh encoder.
    Giống hệt baseline — chỉ encoder thay đổi, decoder học từ đầu.
    """
    if not os.path.exists(weights_path):
        print(f'  ⚠️  Không tìm thấy weights: {weights_path}')
        return
    print(f'  Đang load VGG16-Hybrid weights từ {weights_path} ...')
    try:
        w = np.load(weights_path, allow_pickle=True).item()
    except Exception:
        try:
            import h5py
            w = {}
            with h5py.File(weights_path, 'r') as f:
                for k in f.keys():
                    w[k] = [np.array(f[k][wk]) for wk in f[k].keys()]
        except Exception as e:
            print(f'  ⚠️  Không đọc được weights: {e}')
            return
    suffixes = ['', '_half', '_quarter']
    copied = 0
    for name in VGG_LAYER_NAMES:
        if name not in w:
            continue
        for sfx in suffixes:
            try:
                layer = model.get_layer(name + sfx)
                layer.set_weights(w[name])
                copied += 1
            except (ValueError, KeyError):
                pass
    print(f'  ✅ Copied pretrained weights vào {copied} layers')


In [ ]:
%%writefile /kaggle/working/data_loader.py
"""
tf.data pipeline cho MSI-Net V2 — giữ nguyên hoàn toàn như baseline.
Aspect-ratio preserving resize + symmetric padding (126 RGB, 0 saliency).
"""
import os, tensorflow as tf

def _get_file_list(path):
    files = []
    for root, _, fs in os.walk(path):
        for f in sorted(fs):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                files.append(os.path.join(root, f))
    if not files:
        raise FileNotFoundError(f'Không có ảnh tại: {path}')
    return sorted(files)

def _resize_keep_aspect(image, target_h, target_w):
    cur = tf.shape(image)[:2]
    h_ratio = tf.cast(target_h, tf.float64) / tf.cast(cur[0], tf.float64)
    w_ratio = tf.cast(target_w, tf.float64) / tf.cast(cur[1], tf.float64)
    ratio   = tf.minimum(h_ratio, w_ratio)
    new_h   = tf.cast(tf.round(tf.cast(cur[0], tf.float64) * ratio), tf.int32)
    new_w   = tf.cast(tf.round(tf.cast(cur[1], tf.float64) * ratio), tf.int32)
    new_size = tf.stack([new_h, new_w])
    shrinking = tf.logical_or(cur[0] > new_h, cur[1] > new_w)
    img4d = tf.expand_dims(image, 0)
    resized = tf.cond(
        shrinking,
        lambda: tf.image.resize(img4d, new_size, method='area'),
        lambda: tf.image.resize(img4d, new_size, method='bicubic'))
    return tf.clip_by_value(resized[0], 0.0, 255.0)

def _pad_to_target(image, target_h, target_w):
    cur = tf.shape(image)
    is_rgb = tf.equal(cur[2], 3)
    pad_val = tf.cond(is_rgb, lambda: 126.0, lambda: 0.0)
    pad_v = tf.cast(target_h - cur[0], tf.float32) / 2.0
    pad_h = tf.cast(target_w - cur[1], tf.float32) / 2.0
    padding = [
        [tf.cast(tf.math.floor(pad_v), tf.int32), tf.cast(tf.math.ceil(pad_v), tf.int32)],
        [tf.cast(tf.math.floor(pad_h), tf.int32), tf.cast(tf.math.ceil(pad_h), tf.int32)],
        [0, 0],
    ]
    return tf.pad(image, padding, constant_values=pad_val)

def _parse(img_path, sal_path, h, w):
    img_raw = tf.io.read_file(img_path)
    img = tf.cond(tf.image.is_jpeg(img_raw),
                  lambda: tf.image.decode_jpeg(img_raw, channels=3),
                  lambda: tf.image.decode_png(img_raw,  channels=3))
    img = tf.cast(img, tf.float32)
    img = _resize_keep_aspect(img, h, w)
    img = _pad_to_target(img, h, w)
    img = img / 255.0
    img = tf.ensure_shape(img, [h, w, 3])

    sal_raw = tf.io.read_file(sal_path)
    sal = tf.cond(tf.image.is_jpeg(sal_raw),
                  lambda: tf.image.decode_jpeg(sal_raw, channels=1),
                  lambda: tf.image.decode_png(sal_raw,  channels=1))
    sal = tf.cast(sal, tf.float32)
    sal = _resize_keep_aspect(sal, h, w)
    sal = _pad_to_target(sal, h, w)
    sal = sal / 255.0
    sal = tf.ensure_shape(sal, [h, w, 1])

    img_half    = tf.image.resize(img, [h//2, w//2], method='area')
    img_quarter = tf.image.resize(img, [h//4, w//4], method='area')
    return (img, img_half, img_quarter), sal

def build_dataset(img_dir, sal_dir, target_size, batch_size,
                  shuffle=True, buffer_size=1000):
    imgs = _get_file_list(img_dir)
    sals = _get_file_list(sal_dir)
    assert len(imgs) == len(sals), f'Mismatch: {len(imgs)} imgs vs {len(sals)} sals'
    h, w = target_size
    ds = tf.data.Dataset.from_tensor_slices((imgs, sals))
    if shuffle:
        ds = ds.shuffle(buffer_size, seed=42)
    ds = ds.map(lambda x, y: _parse(x, y, h, w),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(imgs)


In [ ]:
%%writefile /kaggle/working/metrics.py
"""
Evaluation metrics: KLD, CC, SIM, NSS, AUC-Judd — giống baseline.
"""
import numpy as np

def _normalize(x, eps=1e-7):
    s = x.sum(); return x / (s + eps)

def kld_metric(gt, pred, eps=1e-7):
    gt   = _normalize(gt.astype(np.float64))
    pred = _normalize(pred.astype(np.float64))
    return float(np.sum(gt * np.log(eps + gt / (eps + pred))))

def cc_metric(gt, pred, eps=1e-7):
    gt   = gt.astype(np.float64).ravel(); pred = pred.astype(np.float64).ravel()
    gt  -= gt.mean();  pred -= pred.mean()
    denom = np.sqrt((gt**2).sum() * (pred**2).sum()) + eps
    return float(np.sum(gt * pred) / denom)

def sim_metric(gt, pred):
    gt   = _normalize(gt.astype(np.float64))
    pred = _normalize(pred.astype(np.float64))
    return float(np.sum(np.minimum(gt, pred)))

def nss_metric(gt_fixations, pred, eps=1e-7):
    pred = pred.astype(np.float64)
    pred = (pred - pred.mean()) / (pred.std() + eps)
    fix  = gt_fixations.astype(bool)
    if fix.sum() == 0: return 0.0
    return float(pred[fix].mean())

def auc_judd(gt_fixations, pred):
    pred = pred.astype(np.float64).ravel()
    fix  = gt_fixations.astype(bool).ravel()
    if fix.sum() == 0 or (~fix).sum() == 0: return 0.5

    pos_scores = pred[fix]
    neg_scores = pred[~fix]

    # Dùng sorting — O(n log n), không tạo matrix lớn
    n_pos = len(pos_scores)
    n_neg = len(neg_scores)

    all_scores = np.concatenate([pos_scores, neg_scores])
    labels     = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])

    sorted_idx = np.argsort(all_scores)[::-1]
    labels_sorted = labels[sorted_idx]

    tps = np.cumsum(labels_sorted) / n_pos
    fps = np.cumsum(1 - labels_sorted) / n_neg

    tps = np.concatenate([[0.0], tps])
    fps = np.concatenate([[0.0], fps])

    try:    return float(np.trapezoid(tps, fps))
    except: return float(np.trapz(tps, fps))

def evaluate_batch(gt_maps, pred_maps, gt_fixations=None):
    results = {k: [] for k in ["KLD", "CC", "SIM", "NSS", "AUC"]}
    for i in range(len(gt_maps)):
        results["KLD"].append(kld_metric(gt_maps[i],  pred_maps[i]))
        results["CC"] .append(cc_metric (gt_maps[i],  pred_maps[i]))
        results["SIM"].append(sim_metric(gt_maps[i],  pred_maps[i]))
        if gt_fixations is not None:
            results["NSS"].append(nss_metric(gt_fixations[i], pred_maps[i]))
            results["AUC"].append(auc_judd  (gt_fixations[i], pred_maps[i]))
    return {k: float(np.mean(v)) for k, v in results.items() if v}


In [ ]:
%%writefile /kaggle/working/download_salicon.py

import os, zipfile
import gdown

def download_salicon(data_path):
    default_path   = os.path.join(data_path, "salicon")
    stimuli_train  = os.path.join(default_path, "stimuli",  "train")
    saliency_train = os.path.join(default_path, "saliency", "train")
    if os.path.exists(stimuli_train) and len(os.listdir(stimuli_train)) > 0:
        print("✅ SALICON đã tồn tại."); return default_path
    for d in [stimuli_train,
              os.path.join(default_path, "stimuli",  "val"),
              saliency_train,
              os.path.join(default_path, "saliency", "val"),
              os.path.join(default_path, "fixations")]:
        os.makedirs(d, exist_ok=True)
    tmp  = os.path.join(data_path, "tmp.zip")
    ids  = [
        "1g8j-hTT-51IG1UFwP0xTGhLdgIUCW5e5",
        "1P-jeZXCsjoKO79OhFUgnj6FGcyvmLDPj",
        "1PnO7szbdub1559LfjYHMy65EDC4VhJC8",
    ]
    paths = [
        os.path.join(default_path, "stimuli"),
        os.path.join(default_path, "fixations"),
        os.path.join(default_path, "saliency"),
    ]
    for i, fid in enumerate(ids):
        print(f"  Tải file {i+1}/3 ...", end="", flush=True)
        gdown.download(f"https://drive.google.com/uc?id={fid}&export=download", tmp, quiet=True)
        with zipfile.ZipFile(tmp, "r") as z:
            for f in z.namelist():
                if "test" not in f: z.extract(f, paths[i])
        os.remove(tmp); print(" done!")
    images_dir = os.path.join(default_path, "stimuli", "images")
    if os.path.exists(images_dir):
        import shutil
        for sub in os.listdir(images_dir):
            shutil.move(os.path.join(images_dir, sub), os.path.join(default_path, "stimuli", sub))
        os.rmdir(images_dir)
    print("✅ SALICON tải xong!"); return default_path


In [ ]:
%%writefile /kaggle/working/download_mit1003.py
import os, zipfile, urllib.request

MIT1003_STIMULI_URL = "https://people.csail.mit.edu/tjudd/WherePeopleLook/ALLSTIMULI.zip"
MIT1003_FIXMAP_URL  = "https://people.csail.mit.edu/tjudd/WherePeopleLook/ALLFIXATIONMAPS.zip"

def download_mit1003(data_path):
    root     = os.path.join(data_path, "mit1003")
    stim_dir = os.path.join(root, "stimuli")
    sal_dir  = os.path.join(root, "saliency")
    fix_dir  = os.path.join(root, "fixations")
    n_imgs = sum(1 for _,_,fs in os.walk(stim_dir)
                 for f in fs if f.lower().endswith((".jpg",".jpeg",".png")))
    if n_imgs >= 1000:
        print("✅ MIT1003 đã tồn tại."); return root
    for d in [stim_dir, sal_dir, fix_dir]: os.makedirs(d, exist_ok=True)
    tmp = os.path.join(data_path, "tmp_mit.zip")
    def _dl(url, dest_dir, skip_suffix=None):
        print(f"  Tải {url.split('/')[-1]} ...", end="", flush=True)
        urllib.request.urlretrieve(url, tmp)
        with zipfile.ZipFile(tmp, "r") as z:
            for name in z.namelist():
                base = os.path.basename(name)
                if not base: continue
                if skip_suffix and not base.lower().endswith(skip_suffix): continue
                with z.open(name) as src, open(os.path.join(dest_dir, base), "wb") as dst:
                    dst.write(src.read())
        os.remove(tmp); print(" done!")
    _dl(MIT1003_STIMULI_URL, stim_dir, skip_suffix=(".jpeg", ".jpg", ".png"))
    _dl(MIT1003_FIXMAP_URL,  sal_dir,  skip_suffix=(".jpg",  ".jpeg", ".png"))
    print("✅ MIT1003 tải xong!"); return root


In [ ]:
%%writefile /kaggle/working/download_cat2000.py
import os, zipfile, time, requests

CAT2000_URL = "http://saliency.mit.edu/trainSet.zip"

def download_cat2000(data_path):
    root     = os.path.join(data_path, "cat2000")
    stim_dir = os.path.join(root, "stimuli")
    if os.path.exists(stim_dir) and len(os.listdir(stim_dir)) >= 20:
        print("✅ CAT2000 đã tồn tại."); return root
    os.makedirs(data_path, exist_ok=True)
    tmp = os.path.join(data_path, "tmp_cat.zip")
    MAX_RETRIES, CHUNK = 5, 1024*1024
    for attempt in range(1, MAX_RETRIES + 1):
        downloaded = os.path.getsize(tmp) if os.path.exists(tmp) else 0
        headers = {"Range": f"bytes={downloaded}-"} if downloaded else {}
        try:
            print(f"  Tải CAT2000 (~1 GB) — lần {attempt} ", end="", flush=True)
            resp = requests.get(CAT2000_URL, headers=headers, stream=True, timeout=(30, 120))
            if resp.status_code == 200 and downloaded: downloaded = 0
            if resp.status_code not in (200, 206): raise RuntimeError(f"HTTP {resp.status_code}")
            mode = "ab" if downloaded else "wb"
            with open(tmp, mode) as f:
                for chunk in resp.iter_content(CHUNK):
                    if chunk: f.write(chunk); print(".", end="", flush=True)
            print(" done!"); break
        except Exception as e:
            print(f" lỗi: {e}")
            if attempt < MAX_RETRIES: time.sleep(5 * attempt)
            else: raise
    print("  Đang giải nén...", end="", flush=True)
    with zipfile.ZipFile(tmp, "r") as z:
        for name in z.namelist():
            if not ("Output" in name or "allFixData" in name): z.extract(name, data_path)
    os.remove(tmp)
    train_set = os.path.join(data_path, "trainSet")
    if os.path.exists(train_set): os.rename(train_set, root)
    for old, new in [("Stimuli","stimuli"),("FIXATIONLOCS","fixations"),("FIXATIONMAPS","saliency")]:
        src = os.path.join(root, old)
        if os.path.exists(src): os.rename(src, os.path.join(root, new))
    print(" done!\n✅ CAT2000 xong!"); return root


## ── Bước 4: Tải dữ liệu ───────────────────────
> Bật **Internet** trong Kaggle Settings trước khi chạy.

In [ ]:
sys.path.insert(0, "/kaggle/working")
import numpy as np
from download_salicon import download_salicon
from download_mit1003 import download_mit1003
from download_cat2000 import download_cat2000

salicon_root = download_salicon(DATA_DIR)
mit1003_root = download_mit1003(DATA_DIR)
cat2000_root = download_cat2000(DATA_DIR)

def _count(p):
    if not os.path.exists(p): return 0
    return sum(1 for _,_,fs in os.walk(p) for f in fs if f.lower().endswith((".jpg",".jpeg",".png")))

for split in ["train","val"]:
    print(f"  SALICON {split}: {_count(os.path.join(salicon_root,'stimuli',split))} ảnh")
print(f"  MIT1003 : {_count(os.path.join(mit1003_root,'stimuli'))} ảnh")
print(f"  CAT2000 : {_count(os.path.join(cat2000_root,'stimuli'))} ảnh (20 danh mục)")

## ── Bước 5: Build model V2 + Load VGG16-Hybrid weights ───

In [ ]:
import tensorflow as tf, numpy as np, sys
for mod in ['config','loss','model','data_loader','metrics']:
    sys.modules.pop(mod, None)

import config, loss as loss_module
from model       import build_msinet, load_vgg16_hybrid_weights
from data_loader import build_dataset

gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

TARGET_SIZE = config.DIMS['image_size_salicon']  # (240, 320)
BATCH_SIZE  = config.PARAMS['batch_size']         # 4
N_EPOCHS    = config.PARAMS['n_epochs']            # 10
LR          = config.PARAMS['learning_rate']       # 1e-5

train_ds, n_train = build_dataset(
    os.path.join(salicon_root, 'stimuli',  'train'),
    os.path.join(salicon_root, 'saliency', 'train'),
    TARGET_SIZE, BATCH_SIZE, shuffle=True)
valid_ds, n_valid = build_dataset(
    os.path.join(salicon_root, 'stimuli',  'val'),
    os.path.join(salicon_root, 'saliency', 'val'),
    TARGET_SIZE, BATCH_SIZE, shuffle=False)

print(f'Train: {n_train} | Val: {n_valid}')

model = build_msinet(input_shape=(*TARGET_SIZE, 3))
model.summary(line_length=90)
print(f'\n✅ MSI-Net V2 — tổng params: {model.count_params():,}')

In [ ]:
# ── Load VGG16-Hybrid pretrained weights (checkpoint từ baseline) ──
CKPT_PREFIX = os.path.join(WEIGHTS_DIR, 'vgg16_hybrid.ckpt')
CKPT_INDEX  = CKPT_PREFIX + '.index'

if not os.path.exists(CKPT_INDEX):
    print('  Đang tải VGG16-Hybrid weights...')
    import gdown, zipfile
    zip_path = os.path.join(WEIGHTS_DIR, 'tmp_vgg.zip')
    gdown.download(
        'https://drive.google.com/uc?id=1ff0va472Xs1bvidCwRlW3Ctf7Hbyyn7p&export=download',
        zip_path, quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(WEIGHTS_DIR)
    os.remove(zip_path)

try:
    reader  = tf.train.load_checkpoint(CKPT_PREFIX)
    var_map = reader.get_variable_to_shape_map()

    VGG_CONV = [
        'conv1_1','conv1_2','conv2_1','conv2_2',
        'conv3_1','conv3_2','conv3_3',
        'conv4_1','conv4_2','conv4_3',
        'conv5_1','conv5_2','conv5_3',
    ]

    copied = 0
    for base in VGG_CONV:
        w_key = f'{base}/weights'
        b_key = f'{base}/biases'
        if w_key not in var_map or b_key not in var_map:
            continue
        kernel = reader.get_tensor(w_key)  # shape [kH, kW, in, out]
        bias   = reader.get_tensor(b_key)

        for sfx in ['', '_half', '_quarter']:
            try:
                layer = model.get_layer(base + sfx)
            except ValueError:
                continue
            if layer.get_weights()[0].shape == kernel.shape:
                layer.set_weights([kernel, bias])
                copied += 1

    print(f'  ✅ Loaded pretrained weights vào {copied} layers')
# Kỳ vọng: 13 conv layers × 3 nhánh = 39
except Exception as e:
    print(f'  ⚠️  Không load được từ checkpoint: {e}')
    print('  → Thử load từ .npy fallback...')
    npy_path = os.path.join(WEIGHTS_DIR, 'vgg16_hybrid.npy')
    if os.path.exists(npy_path):
        load_vgg16_hybrid_weights(model, npy_path)
    else:
        print('  ⚠️  Không có pretrained weights — train từ scratch')

## ── Bước 6: Huấn luyện trên SALICON ─────────────

In [ ]:
CKPT_DIR    = os.path.join(RESULTS_DIR, 'ckpts')
BEST_CKPT   = os.path.join(CKPT_DIR, 'best', 'msinet_v2_salicon_best.weights.h5')
HISTORY_DIR = os.path.join(RESULTS_DIR, 'history')

def to_model_inputs(scales, sal):
    img_full, img_half, img_quarter = scales
    return ({'input_full': img_full,
             'input_half': img_half,
             'input_quarter': img_quarter}, sal)

train_ds_mi = train_ds.map(to_model_inputs)
valid_ds_mi = valid_ds.map(to_model_inputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss=loss_module.kld)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_CKPT, save_weights_only=True, monitor='val_loss',
        mode='min', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.CSVLogger(
        os.path.join(HISTORY_DIR, 'salicon_log.csv'), append=False),
]

history = model.fit(train_ds_mi, epochs=N_EPOCHS,
                    validation_data=valid_ds_mi, callbacks=callbacks, verbose=1)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('MSI-Net V2 — SALICON Training', fontsize=13, fontweight='bold')

axes[0].plot(history.history['loss'],     label='Train KLD', linewidth=2, marker='o', markersize=4)
axes[0].plot(history.history['val_loss'], label='Val KLD',   linewidth=2, marker='s', markersize=4)
axes[0].set_title('KLD Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

gap = [v - t for t, v in zip(history.history['loss'], history.history['val_loss'])]
axes[1].plot(gap, color='tomato', linewidth=2, marker='^', markersize=4)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Val − Train gap'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Gap')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(HISTORY_DIR, 'v2_salicon_loss.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val_loss: {min(history.history["val_loss"]):.4f}')

In [ ]:
import shutil
import os

src = '/kaggle/working/results/ckpts/best/msinet_v2_salicon_best.weights.h5'
dst = '/kaggle/working/msinet_v2_salicon_best.weights.h5'

if os.path.exists(src):
    shutil.copy(src, dst)
    print("✅ Đã copy file ra thư mục gốc thành công!")
else:
    print("❌ File chưa được sinh ra hoặc Commit ngầm chưa nhả file đâu bạn ơi!")

In [ ]:
from IPython.display import FileLink
FileLink(r'msinet_v2_salicon_best.weights.h5')

## ── Bước 7: Đánh giá SALICON val ────────────────

In [ ]:
import importlib, metrics
importlib.reload(metrics)
from metrics import evaluate_batch

In [ ]:
CKPT_DIR    = os.path.join(RESULTS_DIR, 'ckpts')
BEST_CKPT   = os.path.join(CKPT_DIR, 'best', 'msinet_v2_salicon_best.weights.h5')
HISTORY_DIR = os.path.join(RESULTS_DIR, 'history')

def to_model_inputs(scales, sal):
    img_full, img_half, img_quarter = scales
    return ({'input_full': img_full,
             'input_half': img_half,
             'input_quarter': img_quarter}, sal)

valid_ds_mi = valid_ds.map(to_model_inputs)

# Xác nhận checkpoint tồn tại
import os
if os.path.exists(BEST_CKPT):
    print(f'✅ Checkpoint found: {BEST_CKPT}')
else:
    print(f'❌ Không tìm thấy: {BEST_CKPT}')

In [ ]:
from metrics import evaluate_batch

def evaluate_model(m, dataset, dataset_name='', n_batches=None):
    if n_batches:
        dataset = dataset.take(n_batches)

    all_gt   = np.concatenate([sals.numpy()[:,:,:,0] for _, sals in dataset], axis=0)

    all_pred = m.predict(dataset, verbose=1)
    if isinstance(all_pred, (list, tuple)):
        all_pred = all_pred[0]
    all_pred = all_pred[:,:,:,0]

    fix_maps = (all_gt > 0.1).astype(np.float32)
    scores   = evaluate_batch(all_gt, all_pred, gt_fixations=fix_maps)

    print(f'\n{"="*50}')
    print(f'  📊 {dataset_name}')
    print(f'{"="*50}')
    dirs = {"KLD":"↓ lower","CC":"↑ higher","SIM":"↑ higher","NSS":"↑ higher","AUC":"↑ higher"}
    for k, v in scores.items():
        print(f'  {k:<8} {v:>10.4f}   {dirs.get(k,"")}')
    print(f'{"="*50}')
    return scores

model.load_weights(BEST_CKPT)
scores_salicon = evaluate_model(model, valid_ds.cache(), dataset_name='SALICON val')

In [ ]:
import matplotlib.pyplot as plt
def visualize_predictions(m, dataset, dataset_name='', n_samples=4, save_dir=RESULTS_DIR):
    imgs_list, sals_list, preds_list = [], [], []
    for inputs, sals in dataset:
        if isinstance(inputs, (tuple, list)):
            img_full, img_half, img_quarter = inputs
            preds = m.predict({'input_full': img_full, 'input_half': img_half,
                               'input_quarter': img_quarter}, verbose=0)
            imgs_list.append(img_full.numpy())
        elif isinstance(inputs, dict):
            preds = m.predict(inputs, verbose=0)
            imgs_list.append(list(inputs.values())[0].numpy())
        else:
            preds = m.predict(inputs, verbose=0)
            imgs_list.append(inputs.numpy())
        if isinstance(preds, (list, tuple)): preds = preds[0]
        sals_list.append(sals.numpy()); preds_list.append(preds)
        if sum(len(x) for x in imgs_list) >= n_samples: break

    imgs  = np.concatenate(imgs_list,  axis=0)[:n_samples]
    sals  = np.concatenate(sals_list,  axis=0)[:n_samples]
    preds = np.concatenate(preds_list, axis=0)[:n_samples]

    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
    fig.suptitle(f'MSI-Net V2 — {dataset_name}', fontsize=13, fontweight='bold')
    for i in range(n_samples):
        axes[i,0].imshow(imgs[i]);                  axes[i,0].axis('off')
        axes[i,1].imshow(sals[i,:,:,0], cmap='jet'); axes[i,1].axis('off')
        axes[i,2].imshow(preds[i,:,:,0], cmap='jet'); axes[i,2].axis('off')
        if i == 0:
            axes[i,0].set_title('Ảnh gốc')
            axes[i,1].set_title('Ground Truth')
            axes[i,2].set_title('Predicted')
    plt.tight_layout()
    tag = dataset_name.lower().replace(' ','_').replace('/','_')
    plt.savefig(os.path.join(save_dir, 'images', f'v2_pred_{tag}.png'), dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(model, valid_ds, dataset_name='SALICON val', n_samples=4)

## ── Bước 8: Fine-tune trên MIT1003 ─────────────────

In [ ]:
import random, shutil
from data_loader import build_dataset as _build_ds

MIT_SIZE = config.DIMS['image_size_mit1003']   # (360, 360)
MIT_CKPT = os.path.join(CKPT_DIR, 'best', 'msinet_v2_mit1003_best.weights.h5')

sal_src    = os.path.join(mit1003_root, 'saliency')
stim_src   = os.path.join(mit1003_root, 'stimuli')
stim_train = os.path.join(mit1003_root, 'stimuli',  'train')
stim_val   = os.path.join(mit1003_root, 'stimuli',  'val')
sal_train  = os.path.join(mit1003_root, 'saliency', 'train')
sal_val    = os.path.join(mit1003_root, 'saliency', 'val')
for d in [stim_train, stim_val, sal_train, sal_val]:
    os.makedirs(d, exist_ok=True)

sal_map = {}
for f in os.listdir(sal_src):
    if '_fixMap' in f and f.lower().endswith(('.png', '.jpg', '.jpeg')):
        stem = f.replace('_fixMap.jpg','').replace('_fixMap.png','')
        sal_map[stem] = os.path.join(sal_src, f)
print(f'Tìm thấy {len(sal_map)} fixMap files')

all_imgs = sorted([f for f in os.listdir(stim_src)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))
                   and os.path.isfile(os.path.join(stim_src, f))])
if not all_imgs:
    all_imgs = []
    for sp in ['train','val']:
        sp_dir = os.path.join(stim_src, sp)
        if os.path.isdir(sp_dir):
            all_imgs += [f for f in os.listdir(sp_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    all_imgs = sorted(all_imgs)
print(f'Tổng ảnh MIT1003: {len(all_imgs)}')

random.seed(42); random.shuffle(all_imgs)
n_tr = int(len(all_imgs) * 0.9)
split_map = {f: 'train' for f in all_imgs[:n_tr]}
split_map.update({f: 'val' for f in all_imgs[n_tr:]})

def copy_stim_sal(fn, split):
    src_img = os.path.join(stim_src, fn)
    if not os.path.exists(src_img):
        for sp2 in ['train','val']:
            c = os.path.join(stim_src, sp2, fn)
            if os.path.exists(c): src_img = c; break
    dst_img = os.path.join(mit1003_root, 'stimuli', split, fn)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.copy2(src_img, dst_img)
    stem = os.path.splitext(fn)[0]
    if stem in sal_map:
        dst_sal = os.path.join(mit1003_root, 'saliency', split,
                               os.path.basename(sal_map[stem]))
        if not os.path.exists(dst_sal): shutil.copy2(sal_map[stem], dst_sal)
        return True
    return False

tr_ok = tr_miss = v_ok = v_miss = 0
for fn, split in split_map.items():
    found = copy_stim_sal(fn, split)
    if split == 'train': tr_ok += found; tr_miss += not found
    else:                v_ok  += found; v_miss  += not found
print(f'Split — train: {tr_ok} OK / {tr_miss} miss | val: {v_ok} OK / {v_miss} miss')

mit_tr_ds, n_mt = _build_ds(stim_train, sal_train, MIT_SIZE, BATCH_SIZE, shuffle=True)
mit_val_ds, n_mv = _build_ds(stim_val,  sal_val,   MIT_SIZE, BATCH_SIZE, shuffle=False)
print(f'MIT1003 — train: {n_mt} | val: {n_mv}')

In [ ]:
# ── Clone weights sang model MIT_SIZE, fine-tune ───────────────────────
def _clone_weights(dst_model, src_model_or_path):
    if isinstance(src_model_or_path, str):
        tmp = build_msinet(input_shape=(*TARGET_SIZE, 3))
        tmp.load_weights(src_model_or_path)
        src = tmp
    else:
        src = src_model_or_path
    src_map = {l.name: l for l in src.layers}
    copied  = 0
    for layer in dst_model.layers:
        if layer.name in src_map:
            w = src_map[layer.name].get_weights()
            if w:
                try: layer.set_weights(w); copied += 1
                except ValueError: pass
    return copied

mit_model = build_msinet(input_shape=(*MIT_SIZE, 3))
copied = _clone_weights(mit_model, BEST_CKPT)
print(f'✅ Copied {copied} layers từ SALICON → MIT1003 model')

mit_tr_mi  = mit_tr_ds.map(to_model_inputs)
mit_val_mi = mit_val_ds.map(to_model_inputs)

mit_model.compile(optimizer=tf.keras.optimizers.Adam(LR * 0.1),
                  loss=loss_module.kld)
print('\n🔁 Fine-tuning trên MIT1003...')
mit_ft_history = mit_model.fit(
    mit_tr_mi, epochs=10, validation_data=mit_val_mi,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            MIT_CKPT, save_weights_only=True, monitor='val_loss',
            mode='min', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger(
            os.path.join(HISTORY_DIR, 'mit1003_log.csv'), append=False),
    ], verbose=1)
mit_model.load_weights(MIT_CKPT)
print(f'Best val_loss MIT1003: {min(mit_ft_history.history["val_loss"]):.4f}')

## ── Bước 9: Đánh giá MIT1003 val ─────────────────

In [ ]:
scores_mit = evaluate_model(mit_model, mit_val_ds, dataset_name='MIT1003 val')
visualize_predictions(mit_model, mit_val_ds, dataset_name='MIT1003 val', n_samples=4)

## ── Bước 10: Fine-tune trên CAT2000 ───────────

In [ ]:
import random, shutil
from collections import defaultdict

CAT_SIZE = config.DIMS['image_size_cat2000']   # (216, 384)
CAT_CKPT = os.path.join(CKPT_DIR, 'best', 'msinet_v2_cat2000_best.weights.h5')

cat_stim = os.path.join(cat2000_root, 'stimuli')
cat_sal  = os.path.join(cat2000_root, 'saliency')

if os.path.exists(cat_stim) and sum(len(f) for _,_,f in os.walk(cat_stim)) >= 100:
    all_cat_imgs = []
    for subdir in sorted(os.listdir(cat_stim)):
        sub_path = os.path.join(cat_stim, subdir)
        if os.path.isdir(sub_path):
            for fn in sorted(os.listdir(sub_path)):
                if fn.lower().endswith(('.jpg','.jpeg','.png')):
                    all_cat_imgs.append((subdir, fn))

    random.seed(42)
    by_cat = defaultdict(list)
    for cat, fn in all_cat_imgs: by_cat[cat].append(fn)
    train_cat, val_cat = [], []
    for cat, fns in by_cat.items():
        random.shuffle(fns)
        n_tr_c = int(len(fns) * 0.8)
        for fn in fns[:n_tr_c]: train_cat.append((cat, fn))
        for fn in fns[n_tr_c:]: val_cat.append((cat, fn))
    print(f'CAT2000 — train: {len(train_cat)} | val: {len(val_cat)}')

    for split, lst in [('train', train_cat), ('val', val_cat)]:
        for sub in ['stimuli','saliency']:
            os.makedirs(os.path.join(cat2000_root, sub, split), exist_ok=True)
        for cat, fn in lst:
            src_img = os.path.join(cat_stim, cat, fn)
            dst_img = os.path.join(cat2000_root, 'stimuli', split, f'{cat}_{fn}')
            if os.path.exists(src_img) and not os.path.exists(dst_img):
                shutil.copy2(src_img, dst_img)
            for ext in [fn, fn.replace('.jpeg','.jpg'), fn.replace('.png','.jpg')]:
                src_sal = os.path.join(cat_sal, cat, ext)
                if os.path.exists(src_sal):
                    dst_sal = os.path.join(cat2000_root, 'saliency', split, f'{cat}_{ext}')
                    if not os.path.exists(dst_sal): shutil.copy2(src_sal, dst_sal)
                    break

    cat_tr_ds, n_ct = _build_ds(
        os.path.join(cat2000_root,'stimuli','train'),
        os.path.join(cat2000_root,'saliency','train'),
        CAT_SIZE, BATCH_SIZE, shuffle=True)
    cat_v_ds, n_cv = _build_ds(
        os.path.join(cat2000_root,'stimuli','val'),
        os.path.join(cat2000_root,'saliency','val'),
        CAT_SIZE, BATCH_SIZE, shuffle=False)
    print(f'Dataset: train={n_ct} | val={n_cv}')

    cat_model = build_msinet(input_shape=(*CAT_SIZE, 3))
    src_ckpt = MIT_CKPT if os.path.exists(MIT_CKPT) else BEST_CKPT
    copied = _clone_weights(cat_model, src_ckpt)
    print(f'✅ Copied {copied} layers → CAT2000 model')

    cat_tr_mi = cat_tr_ds.map(to_model_inputs)
    cat_v_mi  = cat_v_ds.map(to_model_inputs)
    cat_model.compile(optimizer=tf.keras.optimizers.Adam(LR * 0.01),
                      loss=loss_module.kld)
    print('\n🔁 Fine-tuning trên CAT2000...')
    cat_ft_history = cat_model.fit(
        cat_tr_mi, epochs=10, validation_data=cat_v_mi,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                CAT_CKPT, save_weights_only=True, monitor='val_loss',
                mode='min', save_best_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            tf.keras.callbacks.CSVLogger(
                os.path.join(HISTORY_DIR, 'cat2000_log.csv'), append=False),
        ], verbose=1)
    cat_model.load_weights(CAT_CKPT)
    scores_cat = evaluate_model(cat_model, cat_v_ds, dataset_name='CAT2000 val')
    visualize_predictions(cat_model, cat_v_ds, dataset_name='CAT2000 val', n_samples=4)
else:
    print('⚠️  CAT2000 không tìm thấy — bỏ qua bước này.')
    cat_ft_history = None; scores_cat = {}

## ── Bước 11: Bảng kết quả tổng hợp ──────────────────

In [ ]:
print("\n" + "="*70)
print("📋 KẾT QUẢ ĐÁNH GIÁ — MSI-Net V2 (VGG16-Hybrid + FPN Decoder)")
print("="*70)
header = f"{'Metric':<10} {'SALICON val':>14} {'MIT1003 val':>14} {'CAT2000 val':>14}"
print(header)
print("-"*70)
for k in ["KLD","CC","SIM","NSS","AUC"]:
    s1 = f"{scores_salicon.get(k, float('nan')):.4f}"
    s2 = f"{scores_mit.get(k, float('nan')):.4f}"   if scores_mit else "  N/A"
    s3 = f"{scores_cat.get(k, float('nan')):.4f}"   if scores_cat else "  N/A"
    print(f"{k:<10} {s1:>14} {s2:>14} {s3:>14}")
print("="*70)
print("Note: KLD ↓ better | CC, SIM, NSS, AUC ↑ better")
print()
print("📌 Cải tiến trong V2:")
print("   • Giữ nguyên: VGG16-Hybrid encoder, 3-branch MSI, lr=1e-5, data pipeline")
print("   • Nâng cấp  : Decoder → FPN với skip connections pool1/pool2/pool3")
print("   • Lý thuyết : Skip connections recover spatial detail bị mất qua MaxPool")

## ── Bước 12: Loss curves tổng hợp ─────────────────

In [ ]:
import pandas as pd

logs = {
    'SALICON': os.path.join(HISTORY_DIR, 'salicon_log.csv'),
    'MIT1003': os.path.join(HISTORY_DIR, 'mit1003_log.csv'),
    'CAT2000': os.path.join(HISTORY_DIR, 'cat2000_log.csv'),
}
existing = [(name, path) for name, path in logs.items() if os.path.exists(path)]

if existing:
    fig, axes = plt.subplots(1, len(existing), figsize=(6*len(existing), 4))
    if len(existing) == 1: axes = [axes]
    for ax, (name, path) in zip(axes, existing):
        df = pd.read_csv(path)
        ax.plot(df['loss'],     label='Train', linewidth=2)
        ax.plot(df['val_loss'], label='Val',   linewidth=2)
        ax.set_title(f'MSI-Net V2 — {name}')
        ax.set_xlabel('Epoch'); ax.set_ylabel('KLD Loss')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(HISTORY_DIR, 'v2_all_loss_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

## ── Bước 13: Lưu model & đóng gói kết quả ────────────

In [ ]:
import zipfile, datetime

model.save_weights(os.path.join(RESULTS_DIR, 'v2_salicon_weights.weights.h5'))
print('✅ SALICON model weights saved')

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
zip_path  = f'/kaggle/working/msinet_v2_results_{timestamp}.zip'

INCLUDE_DIRS  = [RESULTS_DIR, WEIGHTS_DIR]
INCLUDE_FILES = [
    '/kaggle/working/config.py',
    '/kaggle/working/loss.py',
    '/kaggle/working/model.py',
    '/kaggle/working/data_loader.py',
    '/kaggle/working/metrics.py',
]

added, skipped = [], []
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base_dir in INCLUDE_DIRS:
        if not os.path.exists(base_dir):
            skipped.append(base_dir); continue
        for root, _, files in os.walk(base_dir):
            for fn in files:
                fp  = os.path.join(root, fn)
                arc = os.path.relpath(fp, '/kaggle/working')
                zf.write(fp, arc); added.append(arc)
    for fp in INCLUDE_FILES:
        if os.path.exists(fp):
            arc = os.path.relpath(fp, '/kaggle/working')
            zf.write(fp, arc); added.append(arc)
        else:
            skipped.append(fp)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'{"="*55}')
print(f'📦 Đóng gói: {os.path.basename(zip_path)}')
print(f'   Kích thước : {size_mb:.2f} MB | Số file: {len(added)}')
print(f'{"="*55}')

from IPython.display import FileLink, display
display(FileLink(zip_path, result_html_prefix='⬇️  Tải về: '))

In [ ]:
from IPython.display import FileLink
FileLink(r'msinet_v2_results_20260607_111847.zip')